In [2]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## Model Architecture

The RNN-POA model has four key components:

1. **Shared BLSTM Encoder** – encodes both question and answer with tied weights
2. **Gaussian Kernel** – propagates question-word influence to neighbouring answer positions
3. **Positional Attention** – combines classical attention with position-aware influence
4. **Manhattan Similarity** – measures question-answer relevance via $\exp(-\|h_q - h_a\|_1)$

### Shared Bidirectional LSTM Encoder

The **SharedBLSTM** module is responsible for encoding both questions and answers into a common vector space using a shared bidirectional LSTM network. By using the same encoder for both inputs, the model ensures consistency in representation, which is essential for similarity-based tasks such as ranking or matching.


### Objective

The primary goal of this module is to:
- Convert sequences of token indices into meaningful contextual representations  
- Ensure that question and answer embeddings are directly comparable  
- Capture both past and future context using a bidirectional architecture  


### Input

- `token_ids`: A padded sequence of token indices  
  - Shape: `(batch_size, sequence_length)`  
- `lengths`: Actual (non-padded) lengths of each sequence  
  - Shape: `(batch_size,)`  

Padding is applied to ensure uniform sequence length within a batch.


### Output

- `hidden_states`: Contextual representation of each token  
  - Shape: `(batch_size, sequence_length, 2 × hidden_dim)`  
- `pooled_representation`: A single vector summarizing the sequence  
  - Shape: `(batch_size, 2 × hidden_dim)`  


### Key Components

#### 1. Embedding Layer
- Uses pre-trained word embeddings (e.g., GloVe)  
- Embeddings are **frozen**, meaning they are not updated during training  
- Converts token indices into dense vector representations  


#### 2. Dropout Layer
- Applied to embeddings to prevent overfitting  
- Randomly drops a fraction of input units during training  


#### 3. Bidirectional LSTM
- Processes the sequence in both forward and backward directions  
- Captures:
  - **Forward context** (past words)
  - **Backward context** (future words)  
- Output dimension is doubled (`2 × hidden_dim`) due to bidirectionality  


#### 4. Packed Sequences
- Used to efficiently process variable-length sequences  
- Prevents computation on padded tokens  
- Improves both speed and accuracy  


#### 5. Masking Mechanism
- Identifies valid (non-padded) tokens  
- Ensures that padding does not affect computations  


#### 6. Mean Pooling
- Aggregates token-level representations into a single vector  
- Only considers valid tokens using the mask  
- Produces a fixed-size representation regardless of sequence length  


### Why Shared Encoder?

Using a shared encoder for both questions and answers:
- Ensures both are mapped into the **same feature space**  
- Reduces the number of trainable parameters  
- Improves generalization and consistency  
- Enables meaningful similarity comparison between encoded sequences  

### Intuition

Each sentence is transformed into:
1. A sequence of context-aware token embeddings  
2. A single vector that summarizes the entire sentence  

This representation can then be used for:
- Similarity computation  
- Ranking tasks  
- Downstream attention mechanisms  


In [3]:
class SharedBLSTM(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=50, dropout=0.2):
        super().__init__()
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embed_matrix, dtype=torch.float32), requires_grad=False
        )  # Freeze pre-trained GloVe

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
            dropout=dropout,
            num_layers=1,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, token_ids, lengths):
        embeds = self.dropout(self.embedding(token_ids))  # (B, L, E)

        # Pack for efficient LSTM processing
        lengths_cpu = lengths.cpu().clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(
            embeds, lengths_cpu, batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        hidden_states, _ = nn.utils.rnn.pad_packed_sequence(
            packed_out, batch_first=True, total_length=token_ids.size(1)
        )

        mask = (token_ids != 0).unsqueeze(-1).float()  # (B, L, 1)
        pooled = (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        return hidden_states, pooled

### Gaussian Kernel for Position-Aware Influence

The key insight of the paper: if question word $q_j$ appears at position $j$ in the
answer, neighbouring answer words are likely relevant too. The influence decays with
distance according to a Gaussian kernel:

$$K(p, q_j) = \exp\!\left(-\frac{(p - q_j)^2}{2\sigma'^{\,2}}\right)$$

where $\sigma' = 0.1$ (set empirically) and $p$ ranges over all answer positions.

The cumulative influence at answer position $p$ is:

$$\hat{d}_p = \sum_{q_j \in Q_a} K(p, q_j)$$

where $Q_a$ is the set of answer positions where a question word appears.
Only positions within the propagation scope $\sigma$ contribute.

In [4]:
def compute_position_influence(a_len, q_positions, max_a_len, sigma_scope=25, sigma_prime=0.1):
    """
    Compute the position-aware influence vector d_hat for one sample.

    Math:
      d_hat[p] = sum_{q_j in Q_a} exp(-(p - q_j)^2 / (2 * sigma'^2))
      Only positions within |p - q_j| <= sigma_scope contribute.

    Args:
        a_len:        actual answer length
        q_positions:  list of answer-indices where question words appear
        max_a_len:    padded answer length
        sigma_scope:  propagation scope (paper: 15-35, default 25)
        sigma_prime:  Gaussian std dev (paper: 0.1)

    Returns:
        d_hat: (max_a_len,) tensor of influence values
    """
    d_hat = torch.zeros(max_a_len)
    if len(q_positions) == 0:
        return d_hat

    for p in range(a_len):
        influence = 0.0
        for qj in q_positions:
            if abs(p - qj) <= sigma_scope:
                influence += math.exp(-((p - qj) ** 2) / (2 * sigma_prime ** 2))
        d_hat[p] = influence

    # Normalise to [0, 1] to prevent scale issues
    if d_hat.max() > 0:
        d_hat = d_hat / d_hat.max()
    return d_hat




In [5]:
def batch_position_influence(a_lens, q_positions_list, max_a_len, sigma_scope=25, sigma_prime=0.1):
    """
    Compute position influence for a whole batch.
    Returns: (batch, max_a_len) tensor.
    """
    batch_d = []
    for i in range(len(a_lens)):
        d = compute_position_influence(
            a_lens[i].item() if torch.is_tensor(a_lens[i]) else a_lens[i],
            q_positions_list[i], max_a_len, sigma_scope, sigma_prime
        )
        batch_d.append(d)
    return torch.stack(batch_d)  # (B, L_a)

### Positional Attention Mechanism

Classical attention computes:
$$\alpha_p = \frac{\exp(s(h^a_p, h^q))}{\sum_k \exp(s(h^a_k, h^q))}$$

The paper augments this with the positional influence $\hat{d}_p$:
$$\tilde{\alpha}_p = \frac{\exp(s(h^a_p, h^q)) \cdot (1 + \hat{d}_p)}{\sum_k \exp(s(h^a_k, h^q)) \cdot (1 + \hat{d}_k)}$$

The attended answer representation is $\tilde{h}^a = \sum_p \tilde{\alpha}_p \, h^a_p$.

In [6]:
class ClassicalAttention(nn.Module):
    """Classical Self-Attention for the Question (based on Yang et al. 2016)"""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden_states, mask):
        # hidden_states: (B, L, 2H)
        u = torch.tanh(self.W(hidden_states))
        scores = self.v(u).squeeze(-1)            # (B, L)
        scores = scores.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(scores, dim=1)
        attended = torch.bmm(alpha.unsqueeze(1), hidden_states).squeeze(1)
        return attended



In [7]:
class PositionalAttention(nn.Module):
    """
    Positional Attention Layer (RNN-POA).
    Implements Formula (7): e(h_j, p_j) = v^T tanh(W_H h_j + W_P p_j + b)
    """
    def __init__(self, lstm_hidden_dim, pos_hidden_dim):
        super().__init__()
        self.W_H = nn.Linear(lstm_hidden_dim, lstm_hidden_dim, bias=True) # bias 'b' included here
        self.W_P = nn.Linear(pos_hidden_dim, lstm_hidden_dim, bias=False)
        self.v = nn.Linear(lstm_hidden_dim, 1, bias=False)

    def forward(self, a_hidden, p_vectors, a_mask):
        """
        a_hidden: (B, L_a, 2H)
        p_vectors: (B, L_a, pos_H)
        a_mask: (B, L_a)
        """
        # Additive MLP attention
        x = self.W_H(a_hidden) + self.W_P(p_vectors)
        scores = self.v(torch.tanh(x)).squeeze(-1)    # (B, L_a)

        scores = scores.masked_fill(a_mask == 0, -1e9)
        alpha = F.softmax(scores, dim=1)              # (B, L_a)

        attended = torch.bmm(alpha.unsqueeze(1), a_hidden).squeeze(1) # (B, 2H)
        return attended

### Full RNN-POA Model

Similarity is computed via Manhattan distance with $\ell_1$ norm:
$$\text{sim}(q, a) = \exp\!\left(-\|h^q - \tilde{h}^a\|_1\right)$$

This similarity score ∈ (0, 1] is used as the probability of relevance.

In [8]:
class RNNPOA(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=50, pos_dim=50, sigma_scope=25, sigma_prime=0.1, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.pos_dim = pos_dim

        self.encoder = SharedBLSTM(embed_matrix, hidden_dim, dropout)
        self.q_attn = ClassicalAttention(2 * hidden_dim)
        self.pos_attn = PositionalAttention(2 * hidden_dim, pos_dim)

        # ── Pre-calculate the Influence Base Matrix K (Section 2.3) ──
        # K(i, u) ~ N(Kernel(u), sigma'^2)
        max_dist = MAX_A_LEN
        K = torch.zeros(pos_dim, max_dist)
        for u in range(max_dist):
            # Formula (2): Kernel(u) = exp(-u^2 / (2 * sigma^2))
            kernel_u = math.exp(-(u**2) / (2 * sigma_scope**2))
            # Formula (3): Sample from Gaussian
            K[:, u] = torch.normal(mean=kernel_u, std=sigma_prime, size=(pos_dim,))

        # Register as buffer so it moves to GPU automatically but isn't updated by optimizer
        self.register_buffer('K', K)

    def _compute_p_vectors(self, a_len, q_pos, max_a_len):
        """Builds the accumulated influence vector p_j for each answer word."""
        B = len(a_len)
        p_vectors = torch.zeros(B, max_a_len, self.pos_dim, device=self.K.device)

        for b in range(B):
            valid_len = a_len[b].item() if torch.is_tensor(a_len[b]) else a_len[b]
            for j in range(valid_len):
                for qp in q_pos[b]:
                    dist = abs(j - qp)
                    if dist < self.K.size(1):
                        p_vectors[b, j] += self.K[:, dist] # Accumulating influence
        return p_vectors

    def forward(self, q_ids, a_ids, q_len, a_len, q_pos):
        # 1. Encode
        q_hidden, _ = self.encoder(q_ids, q_len)       # (B, L_q, 2H)
        a_hidden, _ = self.encoder(a_ids, a_len)       # (B, L_a, 2H)

        # 2. Question Attention
        q_mask = (q_ids != 0).float()
        q_attended = self.q_attn(q_hidden, q_mask)     # (B, 2H)

        # 3. Position-aware Influence Vectors
        p_vectors = self._compute_p_vectors(a_len, q_pos, a_ids.size(1)) # (B, L_a, pos_H)

        # 4. Positional Attention for Answer
        a_mask = (a_ids != 0).float()
        a_attended = self.pos_attn(a_hidden, p_vectors, a_mask)  # (B, 2H)

        # 5. Similarity (Manhattan distance)
        sim = torch.exp(-torch.sum(torch.abs(q_attended - a_attended), dim=1))  # (B,)
        return sim